# Quantum Kernel Methods

Build the quantum fidelity kernel `K(x_i, x_j) = |<phi(x_i)|phi(x_j)>|^2` from an IQP feature map, assemble a kernel matrix for a tiny two-class dataset, and classify a test point with a pure-numpy kernel rule.

**Objectives:**
- Read the fidelity kernel exactly from `state_vector()` via the compute-uncompute circuit (no sampling noise)
- Prove the kernel is a valid similarity: `K(x,x) == 1`, symmetric, and bounded in `[0, 1]`
- Confirm the sampled estimator (`lib.ml.classifiers.quantum_kernel`) matches the exact value within shot noise
- Classify with a numpy kernel-weighted nearest-class-mean rule and report accuracy
- Contrast the IQP-kernel geometry against an angle-encoding kernel

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

from lib.ml.feature_maps import angle_encoding, iqp_encoding
from lib.ml.classifiers import quantum_kernel
from lib.utils.visualization import plot_histogram

np.random.seed(0)  # determinism: this notebook is executed headlessly in CI
device = LocalSimulator()

## 1. The fidelity kernel and compute-uncompute

A quantum kernel never trains the circuit. Instead it uses a fixed feature map $U(x)$ to embed each classical point as a quantum state $|\phi(x)\rangle = U(x)\,|0\rangle$, then measures how *similar* two embeddings are with the squared overlap (fidelity):

$$
K(x_i, x_j) \;=\; \bigl|\langle \phi(x_i)\,|\,\phi(x_j)\rangle\bigr|^2 .
$$

We never have $|\phi\rangle$ as a list of numbers on hardware, so we cannot just dot two vectors. The standard trick is **compute-uncompute**: prepare $|\phi(x_j)\rangle$, then *un*-prepare with $U(x_i)^\dagger$. The combined operator is

$$
U(x_i)^\dagger\,U(x_j)\,|0\rangle ,
$$

and the amplitude of the all-zeros outcome is exactly $\langle 0|\,U(x_i)^\dagger U(x_j)\,|0\rangle = \langle \phi(x_i)|\phi(x_j)\rangle$. Squaring its magnitude gives $K$. If $x_i = x_j$ the two unitaries cancel and we land back on $|0\rangle$ with certainty, so $K(x,x) = 1$.

**Convention reminder:** qcsim is *big-endian* (qubit 0 is the leftmost/most-significant bit), and `state_vector()` returns amplitudes for all $2^n$ basis states. The all-zeros basis state is index `0` regardless of endianness, so the overlap is always `state_vector()[0]`.

In [ ]:
# Build the compute-uncompute circuit for the fidelity kernel.
# We read the kernel EXACTLY from state_vector() -- no shots, fully deterministic.
#
# `lib.ml.classifiers.quantum_kernel` does the SAME construction but estimates K
# by sampling the all-zeros frequency. Here we want the exact value for asserts.

def kernel_circuit(x1, x2, feature_map_fn):
    """Circuit for U(x1)^dagger . U(x2) . |0>, spanning all len(x) qubits."""
    n = len(x1)
    circ = feature_map_fn(x2)                       # prepare |phi(x2)>
    circ.add_circuit(feature_map_fn(x1).adjoint())  # then apply U(x1)^dagger
    # qcsim measures ALL qubits at once: a circuit touching only low qubits has a
    # SHORTER state vector. Force it to span all n qubits with an identity on the
    # last wire so state_vector() has length 2^n and index 0 = all-zeros.
    circ.i(n - 1)
    return circ


def exact_kernel(x1, x2, feature_map_fn):
    """Exact K = |<phi(x1)|phi(x2)>|^2 = |amplitude of |0...0>|^2."""
    sv = kernel_circuit(x1, x2, feature_map_fn).state_vector()
    return float(np.abs(sv[0]) ** 2)


x = np.array([0.5, 1.2])
y = np.array([-0.3, 0.8])
print("IQP feature map (reps=2), 2 qubits")
print(f"  K(x, x) = {exact_kernel(x, x, iqp_encoding):.6f}   (must be 1)")
print(f"  K(x, y) = {exact_kernel(x, y, iqp_encoding):.6f}")
print(f"  K(y, x) = {exact_kernel(y, x, iqp_encoding):.6f}   (must equal K(x, y))")

# --- Exact, deterministic asserts on the kernel's defining properties ---
assert abs(exact_kernel(x, x, iqp_encoding) - 1.0) < 1e-9   # self-similarity is exactly 1
assert abs(exact_kernel(x, y, iqp_encoding)
           - exact_kernel(y, x, iqp_encoding)) < 1e-9        # symmetric: K(a,b)==K(b,a)
assert 0.0 <= exact_kernel(x, y, iqp_encoding) <= 1.0        # a squared overlap lives in [0,1]
print("\nAsserts passed: K(x,x)=1, symmetric, and bounded in [0,1].")

## 2. The sampled estimator matches the exact value

On real hardware we cannot read amplitudes; we run the compute-uncompute circuit for many shots and estimate $K$ as the *frequency* of the all-zeros bitstring. That is exactly what `lib.ml.classifiers.quantum_kernel` does. It is an unbiased estimator of the exact value, with error shrinking like $1/\sqrt{\text{shots}}$.

We use sampling here only to *illustrate* the measurement -- every tight assert in this notebook is pinned to the exact `state_vector()` value, never to raw shots.

In [ ]:
# Compare the exact kernel against the sampled estimator (compute-uncompute + shots).
shots = 4000
exact_xy = exact_kernel(x, y, iqp_encoding)
sampled_xy = quantum_kernel(x, y, iqp_encoding, shots=shots)
print(f"exact   K(x, y) = {exact_xy:.4f}")
print(f"sampled K(x, y) = {sampled_xy:.4f}   ({shots} shots)")
print(f"absolute error  = {abs(sampled_xy - exact_xy):.4f}")

# Loose, CI-stable tolerance: a few standard errors of a Bernoulli frequency.
# This proves the estimator tracks the exact value WITHOUT making a brittle
# assertion on noisy sampling.
assert abs(sampled_xy - exact_xy) < 0.05
print("\nSampled estimate matches the exact kernel within shot noise.")

# Visualize the all-zeros outcome the estimator actually counts.
circ = iqp_encoding(y)
circ.add_circuit(iqp_encoding(x).adjoint())
circ.i(len(x) - 1)
counts = device.run(circ, shots=shots).result().measurement_counts
fig = plot_histogram(counts, title="Compute-uncompute outcomes: P('00') = K(x, y)")
plt.show()

## 3. A tiny two-class dataset (numpy only)

Production kernel code hands $K$ to a classical SVM. To keep this notebook self-contained and Pyodide-friendly we use **no sklearn** -- we hardcode a small 2D dataset by hand and implement the classifier in numpy.

The data is two well-separated blobs in the angle range the feature maps expect: class 0 near $(0.5, 0.5)$ and class 1 near $(2.5, 2.5)$. Eight training points, four test points.

In [ ]:
# Hardcoded 2D, two-class dataset -- no sklearn, no random blobs to keep CI stable.
X_train = np.array([
    [0.4, 0.5], [0.6, 0.3], [0.3, 0.6], [0.5, 0.45],   # class 0 blob
    [2.4, 2.6], [2.6, 2.3], [2.3, 2.7], [2.5, 2.5],     # class 1 blob
])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

X_test = np.array([
    [0.5, 0.4],    # near class 0
    [2.5, 2.4],    # near class 1
    [0.45, 0.65],  # near class 0
    [2.55, 2.35],  # near class 1
])
y_test = np.array([0, 1, 0, 1])

print(f"train: {len(X_train)} points, test: {len(X_test)} points")

fig, ax = plt.subplots(figsize=(5, 5))
for cls, color in [(0, "#232f3e"), (1, "#ff9900")]:
    pts = X_train[y_train == cls]
    ax.scatter(pts[:, 0], pts[:, 1], c=color, s=70, label=f"train class {cls}")
ax.scatter(X_test[:, 0], X_test[:, 1], c="none", edgecolors="crimson",
           s=140, linewidths=2, label="test")
ax.set_xlabel("feature 0"); ax.set_ylabel("feature 1")
ax.set_title("Toy two-class dataset"); ax.legend(loc="center")
plt.show()

## 4. The kernel matrix

The kernel matrix $K_{ij} = K(x_i, x_j)$ is the only thing a kernel method ever sees -- it replaces the raw coordinates entirely. We build it from the exact fidelity kernel so the structure is deterministic.

A valid (Gram) kernel matrix has a unit diagonal and is symmetric; both follow directly from the properties we proved in Section 1. We assert them on the full matrix as a guard.

In [ ]:
def kernel_matrix(A, B, feature_map_fn):
    """Exact fidelity-kernel matrix between rows of A and rows of B."""
    return np.array([[exact_kernel(a, b, feature_map_fn) for b in B] for a in A])

K_train = kernel_matrix(X_train, X_train, iqp_encoding)

# Whole-matrix validity checks (exact => stable in CI):
assert np.allclose(np.diag(K_train), 1.0)        # unit diagonal: every point self-similar
assert np.allclose(K_train, K_train.T)           # symmetric Gram matrix
assert K_train.min() >= -1e-12 and K_train.max() <= 1.0 + 1e-12  # all entries in [0,1]
print("Kernel matrix is valid: unit diagonal, symmetric, entries in [0, 1].")

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(K_train, cmap="magma", vmin=0, vmax=1)
ax.set_title("IQP-kernel matrix (train x train)")
ax.set_xlabel("point j"); ax.set_ylabel("point i")
fig.colorbar(im, ax=ax, label="K(x_i, x_j)")
plt.show()
print("\nThe two bright 4x4 blocks are the within-class similarities;")
print("the dark off-diagonal blocks show the classes barely overlap in feature space.")

## 5. Classify with a numpy kernel rule

Production code would hand $K$ to a classical SVM (`sklearn.svm.SVC(kernel='precomputed')`). Here we implement a transparent numpy classifier instead: **kernel-weighted nearest-class-mean**. For a test point $x$ we compute its mean kernel similarity to each class,

$$
s_c(x) \;=\; \frac{1}{|C_c|}\sum_{x_j \in C_c} K(x, x_j),
$$

and predict the class with the larger mean similarity. Because $K$ is the only thing this rule touches, it is genuinely working in the quantum feature space -- not in the raw 2D coordinates.

In [ ]:
def predict_nearest_class_mean(X_query, X_train, y_train, feature_map_fn):
    """Predict each query point's class by larger mean kernel similarity."""
    K_qt = kernel_matrix(X_query, X_train, feature_map_fn)  # (n_query, n_train)
    mean_sim_0 = K_qt[:, y_train == 0].mean(axis=1)
    mean_sim_1 = K_qt[:, y_train == 1].mean(axis=1)
    return (mean_sim_1 > mean_sim_0).astype(int)

pred = predict_nearest_class_mean(X_test, X_train, y_train, iqp_encoding)
accuracy = float((pred == y_test).mean())
print("IQP quantum kernel, nearest-class-mean classifier")
print(f"  predictions = {pred.tolist()}")
print(f"  truth       = {y_test.tolist()}")
print(f"  accuracy    = {accuracy:.2f}")
assert accuracy == 1.0  # this well-separated toy set is perfectly classified
print("\nAll test points classified correctly using only the quantum kernel.")

## 6. IQP geometry vs angle-encoding geometry

The kernel is only as good as the feature map. The IQP map injects feature *products* via ZZ interactions, producing a richer, more nonlinear embedding than plain angle encoding (one $R_y$ rotation per feature). We compare the two by how sharply each separates the classes in kernel space: the gap between the average **within-class** similarity and the average **cross-class** similarity. A bigger gap means a cleaner block structure for a downstream classifier.

In [ ]:
def class_block_means(X, y, feature_map_fn):
    M = kernel_matrix(X, X, feature_map_fn)
    within = M[np.ix_(y == 0, y == 0)].mean()
    cross = M[np.ix_(y == 0, y == 1)].mean()
    return within, cross

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, fmap) in zip(axes, [("IQP", iqp_encoding), ("angle", angle_encoding)]):
    M = kernel_matrix(X_train, X_train, fmap)
    within, cross = class_block_means(X_train, y_train, fmap)
    acc = float((predict_nearest_class_mean(X_test, X_train, y_train, fmap) == y_test).mean())
    im = ax.imshow(M, cmap="magma", vmin=0, vmax=1)
    ax.set_title(f"{name}: within={within:.2f} cross={cross:.2f}\n"
                 f"separation gap={within - cross:.2f}, test acc={acc:.2f}")
    ax.set_xlabel("j"); ax.set_ylabel("i")
fig.colorbar(im, ax=axes, label="K", fraction=0.046)
plt.show()

iqp_w, iqp_c = class_block_means(X_train, y_train, iqp_encoding)
ang_w, ang_c = class_block_means(X_train, y_train, angle_encoding)
print(f"IQP   separation gap = {iqp_w - iqp_c:.3f}")
print(f"angle separation gap = {ang_w - ang_c:.3f}")
# Both separate this easy dataset; the IQP map drives the cross-class similarity
# closer to zero, giving a sharper block structure. This is geometry, not noise:
assert iqp_c < ang_c   # IQP pushes different-class points further apart in kernel space
print("\nIQP pushes cross-class similarity nearer 0 -> a sharper kernel geometry.")

## Exercises

Work these in fresh cells. No solutions are provided.

In [ ]:
# Exercise 1: Vary the IQP encoding depth.
# `iqp_encoding(features, reps=R)` repeats the H / Rz / ZZ block R times.
# TODO: rebuild K_train for reps in (1, 2, 3) and compare the within-vs-cross
#       separation gap. Does more depth always help, or does the kernel
#       saturate (or concentrate toward 0, an early sign of the kernel-
#       concentration problem that limits deep quantum kernels)?
#
# from functools import partial
# for R in (1, 2, 3):
#     fmap = partial(iqp_encoding, reps=R)
#     ...

In [ ]:
# Exercise 2: A harder, non-separable dataset.
# TODO: hardcode (numpy only) two interleaved groups -- e.g. an inner cluster of
#       class 0 surrounded by class-1 points -- where angle encoding's kernel
#       cannot separate the classes but the IQP kernel's product features can.
#       Recompute accuracy for both feature maps with
#       predict_nearest_class_mean and report which map wins.
#
# X_hard = np.array([...])
# y_hard = np.array([...])
# ...

## Summary

- The **fidelity kernel** $K(x_i,x_j)=|\langle\phi(x_i)|\phi(x_j)\rangle|^2$ is computed with a **compute-uncompute** circuit $U(x_i)^\dagger U(x_j)|0\rangle$; the all-zeros amplitude *is* the overlap, read exactly from `state_vector()[0]`.
- Read this way the kernel is provably valid: `K(x,x)=1`, symmetric, and bounded in `[0,1]` -- the asserts here are exact and deterministic.
- `lib.ml.classifiers.quantum_kernel` estimates the same value by **sampling** the all-zeros frequency; it matches the exact value within shot noise ($\sim 1/\sqrt{\text{shots}}$).
- The **kernel matrix** is the only object a kernel method sees. We classified a test point with a pure-numpy **kernel-weighted nearest-class-mean** rule (production code would instead pass $K$ to a classical SVM such as `sklearn.svm.SVC(kernel='precomputed')`).
- The **feature map sets the geometry**: the IQP map's ZZ feature-products drive cross-class similarity nearer zero than plain angle encoding, giving a sharper block structure.

**Next:** [`03-variational-classifier.ipynb`](03-variational-classifier.ipynb) -- instead of a fixed kernel, *train* the circuit end-to-end as a variational quantum classifier.